In [1]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install -U langchain langchain-community langchain-core langchain-huggingface faiss-cpu sentence-transformers flask flask-cors pyngrok transformers accelerate

Found existing installation: langchain 1.4.3
Uninstalling langchain-1.4.3:
  Successfully uninstalled langchain-1.4.3
Found existing installation: langchain-core 1.6.6
Uninstalling langchain-core-1.6.6:
  Successfully uninstalled langchain-core-1.6.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
  

In [ ]:
import torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ==========================================
# 1. CARREGAR MODELO (IA)
# ==========================================
print("Baixando e carregando o modelo Qwen...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2, # Subimos um pouquinho para a IA ser mais natural nas saudações
    return_full_text=False
)

# ==========================================
# 2. BASE VETORIAL (RAG MANUAL)
# ========================================Manaus
35°c
36°27°
Sol com névoa
==
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ADICIONADO: Um texto introdutório geral sobre segurança do paciente
documentos_saude = [
    "Conceito Geral: A Segurança do Paciente é a redução, a um mínimo aceitável, do risco de dano desnecessário associado ao cuidado de saúde. O objetivo é promover uma cultura de segurança, melhorar processos e reduzir riscos nas instituições de saúde.",
    "Protocolo 1: Identificação Correta do Paciente. Exige no mínimo dois identificadores (nome completo do paciente e data de nascimento).",
    "Protocolo 2: Prescrição e Medicamentos. Seguir os 9 certos na administração de medicamentos para evitar erros.",
    "Protocolo 3: Comunicação Efetiva entre profissionais.",
    "Protocolo 4: Cirurgia Segura. Verificação de local, procedimento e paciente corretos.",
    "Protocolo 5: Higiene das Mãos. Previne infecções cruzadas e deve ser feita antes e depois de tocar o paciente.",
    "Protocolo 6: Prevenção de Quedas e Lesões. Uso da Escala de Morse e Braden para avaliar riscos."
]

vector_db = FAISS.from_texts(documentos_saude, embeddings)

# ==========================================
# 3. SERVIDOR FLASK E LÓGICA RAG
# ==========================================
app = Flask(__name__)
CORS(app)

@app.route('/api/chat', methods=['POST'])
def chat():
    dados = request.json
    pergunta = dados.get('message', '')

    if not pergunta:
        return jsonify({'reply': 'Nenhuma mensagem recebida.'}), 400

    try:
        docs_relevantes = vector_db.similarity_search(pergunta, k=2)
        contexto_extraido = "\n".join([doc.page_content for doc in docs_relevantes])

        # ATUALIZADO: Regras mais flexíveis para permitir saudações e conversa básica
        messages = [
            {
                "role": "system",
                "content": "Você é um assistente virtual acolhedor especializado em Segurança do Paciente. Se o usuário fizer uma saudação (como 'oi', 'olá', 'bom dia'), seja educado, cumprimente de volta e pergunte como pode ajudar. Para perguntas sobre saúde, responda APENAS com base no contexto fornecido. Se a dúvida técnica não estiver no contexto, diga: 'Desculpe, não tenho essa informação nos meus protocolos oficiais.'"
            },
            {
                "role": "user",
                "content": f"Contexto Oficial:\n{contexto_extraido}\n\nMensagem do usuário: {pergunta}"
            }
        ]

        resultado = pipe(messages)
        resposta_ia = resultado[0]['generated_text'].strip()

        return jsonify({'reply': resposta_ia})

    except Exception as e:
        print(f"Erro: {e}")
        return jsonify({'reply': 'Desculpe, ocorreu um erro interno no processamento da IA.'}), 500

# ==========================================
# 4. NGROK
# ==========================================
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
public_url = ngrok.connect(5000).public_url

print("\n" + "="*60)
print(f"✅ SERVIDOR ATIVO COM SAUDAÇÕES LIBERADAS!")
print(f"🔗 URL para o seu arquivo script.js: {public_url}/api/chat")
print("="*60 + "\n")

app.run(port=5000)

/tmp/ipykernel_3345/2670518635.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Baixando e carregando o modelo Qwen...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ SERVIDOR ATIVO COM SAUDAÇÕES LIBERADAS!
🔗 URL para o seu arquivo script.js: https://whooping-renovator-keg.ngrok-free.dev/api/chat

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/Oct/2026 17:53:02] "OPTIONS /api/chat HTTP/1.1" 200 -
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
INFO:werkzeug:127.0.0.1 - - [0